<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/notebooks/03_Concentracion_y_estructura_del_comercio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Cuaderno 3. Concentración, diversificación y estructura del comercio

**Asignatura:** Inteligencia en Negocios Globales
**Semana 5 — Nivel intermedio**
**Documento base:** *Métricas de Comercio Exterior e Inteligencia de Negocios Globales. Documento 2 de 3 — Nivel intermedio: concentración, comercio intraindustrial y ventaja comparativa revelada* (Serie de recursos, 2026)

---

## Retomamos el caso

Una empresa floricultora colombiana exporta **claveles frescos** (**HS 060312**). Hoy depende de **Estados Unidos** y quiere diversificar. El candidato sobre la mesa es **Corea del Sur**.

El Cuaderno 2 cerró con seis preguntas que sus métricas no podían responder. Tres de ellas son de este cuaderno:

- ¿Qué tan concentradas están realmente las exportaciones colombianas, medido con rigor y no con el índice que ya viene calculado en el archivo?
- ¿Quién le vende claveles a Corea del Sur, y qué tan atrincherado está ese proveedor?
- ¿La diversificación de la canasta exportadora colombiana es estructural o es más variedad del mismo sector de siempre?

Y este cuaderno trae además dos fuentes nuevas: la **canasta exportadora completa de Colombia** (97 capítulos del Sistema Armonizado) y los **registros aduaneros de la DIAN** vía Legiscomex, 273.033 declaraciones de exportación de claveles entre 2020 y 2026.

## Las tres métricas de este cuaderno

| Métrica | Pregunta que responde | Rango |
|---|---|---|
| **Índice de Herfindahl-Hirschman (HHI)** | ¿De cuántos clientes —o de cuántos productos— depende realmente este negocio? | de 0 a 1 |
| **Índice de Theil** | ¿La diversificación que veo es estructural o es apariencia? | de 0 a ∞, descomponible |
| **Grubel-Lloyd** | ¿Este comercio es competencia real o intercambio dentro de una misma cadena? | de 0 a 1 |

Cada una se presenta con la misma estructura de los cuadernos anteriores:

1. **Qué es**
2. **Para qué sirve y cómo se usa**
3. **La fórmula y su explicación matemática detallada**
4. **Ejemplo numérico paso a paso**
5. **Cálculo sobre nuestros datos reales**
6. **Interpretación y umbrales**
7. **Impacto en el negocio y la decisión que habilita**
8. **De dónde se saca exactamente el dato**

## Cómo usar este cuaderno

Ejecuta las celdas **en orden**, de arriba hacia abajo. El cuaderno viene configurado en modo `"github"`: descarga sus propios datos del repositorio del curso, así que **no tienes que subir nada ni ejecutar antes el Cuaderno 1**.

Los bloques marcados **Para pensar** no tienen código. Son las preguntas que separan a quien calcula un índice de quien lo entiende.

> **Si la celda de descarga te muestra "servidor ocupado" y espera unos segundos, no pasa nada.** GitHub limita cuántas peticiones acepta por dirección IP, y en Colab miles de personas comparten la misma. La celda reintenta sola. Si aun así falla, espera un minuto y vuelve a ejecutarla: los archivos ya descargados no se vuelven a pedir.

---
# 0. Preparación del entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

# Paleta de colores del curso (validada para lectura accesible en pantalla e impresion)
AZUL      = "#2a78d6"    # serie principal / valores positivos
ROJO      = "#e34948"    # valores negativos (par divergente con el azul)
NARANJA   = "#eb6834"    # elemento destacado
VERDE     = "#3f8f6b"    # segunda serie categorica
GRIS_MID  = "#c3c2b7"    # punto neutro de la escala divergente
TINTA     = "#0b0b0b"
GRIS_TEXT = "#52514e"
GRIS_EJE  = "#898781"
REJILLA   = "#e1e0d9"

print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ============================================================
#  CONFIGURACION: cambia solo esta celda
# ============================================================

MODO = "github"         # opciones: "github" | "subir" | "drive" | "local"

URL_DATOS = "https://raw.githubusercontent.com/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/main/data/"

# Los diez archivos que necesita este cuaderno
ARCHIVOS_REPO = [
    # Trade Map - canasta exportadora e importadora de Colombia y del mundo (HS 2 digitos)
    "co_exp_productos_hs2_serie.xls",
    "co_imp_productos_hs2_serie.csv",
    "mundo_exp_productos_hs2_serie.csv",
    "mundo_imp_productos_hs2_serie.csv",
    "co_exp_productos_hs2_indicadores.xls",
    # Trade Map - el producto del caso
    "kor_060312_proveedores_indicadores.csv",
    "kor_060312_proveedores_serie.csv",
    "exporting-countries-in-2025_060312.csv",
    "importing-countries-in-2025_060312.csv",
    # Legiscomex - registros aduaneros DIAN, ya agregados por destino
    "co_060312_destinos_legiscomex.csv",
    "co_060312_auditoria_descargas_legiscomex.csv",
]

RUTA_DRIVE = "/content/drive/MyDrive/Semana 5"    # solo si MODO = "drive"
RUTA_LOCAL = "."                                   # solo si MODO = "local"
CARPETA_SALIDA = "datos_limpios"

print(f"Modo seleccionado: {MODO}")

In [ ]:
# ============================================================
#  Ejecuta esta celda tal cual: prepara la ruta segun el modo
# ============================================================

import io
import time
import shutil
import zipfile
import urllib.request
import urllib.error

REPO_CURSO = "YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales"
RAMA = "main"


def pedir(url, intentos=5):
    '''Descarga una URL, reintentando si el servidor pide esperar.

    GitHub limita las peticiones por direccion IP, y en Colab miles de personas
    comparten la misma direccion. Cuando se pasa el limite responde
    "HTTP 429 Too Many Requests". No es un error del cuaderno: es una cola,
    y se resuelve esperando un poco y volviendo a pedir.
    '''
    for intento in range(intentos):
        try:
            peticion = urllib.request.Request(url, headers={"User-Agent": "cuaderno-ean"})
            with urllib.request.urlopen(peticion, timeout=90) as respuesta:
                return respuesta.read()

        except urllib.error.HTTPError as error:
            if error.code not in (403, 429, 500, 502, 503) or intento == intentos - 1:
                raise
            # Si el servidor dice cuanto esperar le hacemos caso; si no, duplicamos la espera
            espera = int(error.headers.get("Retry-After") or 0) or 2 ** intento
            print(f"    servidor ocupado (HTTP {error.code}); reintento en {espera} s")
            time.sleep(espera)

        except urllib.error.URLError:
            if intento == intentos - 1:
                raise
            time.sleep(2 ** intento)


def descargar_datos(archivos, destino):
    '''Trae los datos del repositorio del curso a la carpeta destino.

    Baja UN solo archivo comprimido con todo el repositorio, en lugar de pedir
    los archivos uno por uno: varias peticiones seguidas disparan el limite de
    GitHub, una sola no. Si el comprimido falla, recurre a bajarlos de a uno.

    Los archivos que ya existen no se vuelven a descargar, asi que puedes
    volver a ejecutar esta celda sin gastar peticiones.
    '''
    os.makedirs(destino, exist_ok=True)
    faltan = [a for a in archivos if not os.path.exists(os.path.join(destino, a))]

    if not faltan:
        print(f"Los {len(archivos)} archivos ya estaban descargados.")
        return

    try:
        print(f"Descargando {len(faltan)} archivos en una sola peticion...\n")
        comprimido = pedir(f"https://codeload.github.com/{REPO_CURSO}/zip/refs/heads/{RAMA}")

        with zipfile.ZipFile(io.BytesIO(comprimido)) as paquete:
            for miembro in paquete.namelist():
                nombre = os.path.basename(miembro)
                if nombre in faltan:
                    with paquete.open(miembro) as origen, \
                         open(os.path.join(destino, nombre), "wb") as salida:
                        shutil.copyfileobj(origen, salida)
                    print(f"  extraido: {nombre}")

    except Exception as error:
        print(f"\n  El paquete fallo ({type(error).__name__}). Voy archivo por archivo.\n")
        for nombre in faltan:
            with open(os.path.join(destino, nombre), "wb") as salida:
                salida.write(pedir(URL_DATOS + nombre))
            print(f"  descargado: {nombre}")
            time.sleep(0.5)                      # un respiro entre peticiones

    perdidos = [a for a in archivos if not os.path.exists(os.path.join(destino, a))]
    if perdidos:
        raise FileNotFoundError(
            f"No se pudieron descargar: {perdidos}\n"
            f"Espera un minuto y vuelve a ejecutar esta celda: los ya bajados no se repiten."
        )


if MODO == "github":
    RUTA_BASE = "datos_crudos"
    descargar_datos(ARCHIVOS_REPO, RUTA_BASE)

elif MODO == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_BASE = RUTA_DRIVE

elif MODO == "subir":
    from google.colab import files
    print("Selecciona los once archivos de datos.\n")
    files.upload()
    RUTA_BASE = "/content"

else:  # local
    RUTA_BASE = RUTA_LOCAL

os.makedirs(CARPETA_SALIDA, exist_ok=True)
print("\nRuta base de los datos:", RUTA_BASE)
print("Carpeta de salida      :", os.path.abspath(CARPETA_SALIDA))

In [ ]:
def buscar_archivo(patron, ruta_base=None):
    '''Busca un archivo por patron dentro de la ruta base, incluyendo subcarpetas.'''
    if ruta_base is None:
        ruta_base = RUTA_BASE

    coincidencias = sorted(glob.glob(os.path.join(ruta_base, "**", patron), recursive=True))

    if len(coincidencias) == 0:
        raise FileNotFoundError(
            f"No encontre ningun archivo que coincida con '{patron}' dentro de '{ruta_base}'."
        )
    if len(coincidencias) > 1:
        print(f"  Aviso: hay {len(coincidencias)} archivos que coinciden con '{patron}'. Uso el primero.")

    return coincidencias[0]


ARCHIVOS = {
    "co_exp":        buscar_archivo("co_exp_productos_hs2_serie.xls"),
    "co_imp":        buscar_archivo("co_imp_productos_hs2_serie.csv"),
    "mundo_exp":     buscar_archivo("mundo_exp_productos_hs2_serie.csv"),
    "mundo_imp":     buscar_archivo("mundo_imp_productos_hs2_serie.csv"),
    "co_exp_ind":    buscar_archivo("co_exp_productos_hs2_indicadores.xls"),
    "kor_prov_ind":  buscar_archivo("kor_060312_proveedores_indicadores.csv"),
    "kor_prov_serie": buscar_archivo("kor_060312_proveedores_serie.csv"),
    "tm_exp_060312": buscar_archivo("exporting-countries-*_060312.csv"),
    "tm_imp_060312": buscar_archivo("importing-countries-*_060312.csv"),
    "legis_destinos": buscar_archivo("co_060312_destinos_legiscomex.csv"),
    "legis_auditoria": buscar_archivo("co_060312_auditoria_descargas_legiscomex.csv"),
}

for nombre, ruta in ARCHIVOS.items():
    print(f"{nombre:16s} -> {os.path.basename(ruta)}")

---
# 1. Los datos nuevos y las tres trampas que traen

El Cuaderno 1 enseñó que un archivo oficial nunca llega listo para usar. Estos tampoco. Antes de calcular un solo índice hay que desarmar tres trampas nuevas, distintas de las nueve del Cuaderno 1.

## 1.1. La primera trampa: el archivo `.xls` que no es un Excel

Trade Map ofrece descargar la canasta exportadora en formato Excel. El archivo llega con extensión `.xls`. Si intentas abrirlo como Excel, falla.

In [ ]:
# Lo intentamos como Excel, que es lo que la extension promete.
# Lo envolvemos en try/except para poder LEER el error en vez de que el cuaderno se detenga.
try:
    prueba = pd.read_excel(ARCHIVOS["co_exp"])
    print("Se abrio como Excel:", prueba.shape)
except Exception as error:
    print("FALLO al abrir como Excel.")
    print("Tipo de error :", type(error).__name__)
    print("Mensaje       :", str(error)[:200])

In [ ]:
# Miremos los primeros caracteres del archivo, como texto plano.
with open(ARCHIVOS["co_exp"], encoding="utf-8") as f:
    print(f.read(220))

El archivo no es un Excel: es una **tabla HTML** con la extensión cambiada. Es un formato heredado que Trade Map conserva por compatibilidad, y es una de las razones por las que tanta gente termina copiando y pegando a mano.

pandas sabe leer tablas HTML con `read_html()`. Devuelve una **lista** de todas las tablas que encuentra en el documento, no una sola: hay que elegir la que tiene los datos.

In [ ]:
tablas = pd.read_html(ARCHIVOS["co_exp"])

print(f"read_html encontro {len(tablas)} tablas en el archivo:")
for i, t in enumerate(tablas):
    print(f"  tabla {i}: {t.shape[0]} filas x {t.shape[1]} columnas")

print("\nLas primeras son decorativas (encabezados de la pagina). La buena es la mas grande.")

## 1.2. La segunda trampa: los códigos vienen con un apóstrofe

Es el mismo problema de los ceros a la izquierda del Cuaderno 1, pero al revés. Allá la fuente entregaba `060312` y pandas lo convertía en el número `60312`, perdiendo el cero. Aquí Trade Map **se adelanta** al problema y escribe `'06` con un apóstrofe delante, que es la marca con la que Excel fuerza a que algo se lea como texto.

El resultado es que el cero sobrevive, pero el apóstrofe se queda pegado al dato. Si no lo quitas, el código `'06` nunca va a coincidir con el `06` del archivo mundial, y el cruce de tablas devolverá vacío sin decirte por qué.

In [ ]:
tabla_grande = max(tablas, key=lambda t: t.shape[0])

print("Asi vienen los codigos, en crudo:")
print(tabla_grande.iloc[:6, 0].tolist())
print()
print("Y asi los necesitamos, para poder cruzarlos con los demas archivos:")
print([str(c).strip().lstrip("'") for c in tabla_grande.iloc[:6, 0]])

## 1.3. La tercera trampa: los miles vienen separados por coma

En los archivos `.csv` de Trade Map el valor `25.609.329.378` llega escrito como `"25,609,329,378"`. Para pandas eso es **texto**, no un número. Si lo sumas, concatena en vez de sumar; si lo ordenas, ordena alfabéticamente y `"9"` queda después de `"25"`.

Y hay una cuarta, menor pero igual de letal: los archivos de **lista de socios** —los que traen país por país en vez de producto por producto— empiezan con una fila de navegación de la página web (`"Go to..."`) **antes** del encabezado real. Si no la saltas, pandas toma esa fila como nombres de columna.

Escribimos las funciones que resuelven las cuatro trampas de una vez, para no repetir el arreglo en cada archivo.

In [ ]:
def a_numero(serie):
    '''Convierte a numero una columna de Trade Map que viene como texto.

    Quita separadores de miles, simbolos y espacios. Lo que no se puede
    convertir queda como NaN (dato faltante) en vez de romper el proceso.
    '''
    return pd.to_numeric(
        pd.Series(serie).astype(str)
                        .str.replace(",", "", regex=False)
                        .str.replace(r"[^0-9.\-]", "", regex=True)
                        .replace("", np.nan),
        errors="coerce",
    )


def limpiar_codigo(serie):
    '''Deja el codigo de capitulo HS listo para cruzar: sin apostrofe y sin espacios.'''
    return serie.astype(str).str.strip().str.lstrip("'").str.strip()


ANIOS = [str(a) for a in range(2021, 2026)]


def leer_canasta(ruta):
    '''Lee un archivo de Trade Map con la canasta por capitulo HS y cinco anios.

    Funciona igual si el archivo es un .csv de verdad o un .xls que en realidad
    es HTML: detecta el formato por el contenido, no por la extension.
    '''
    with open(ruta, encoding="utf-8", errors="ignore") as f:
        primeras_letras = f.read(200).lstrip().lower()

    if primeras_letras.startswith("<"):                      # es HTML disfrazado
        tabla = max(pd.read_html(ruta), key=lambda t: t.shape[0])
    else:                                                    # es un CSV de verdad
        tabla = pd.read_csv(ruta)

    tabla = tabla.iloc[:, -7:]                               # codigo, nombre y los cinco anios
    tabla.columns = ["codigo", "producto"] + ANIOS
    tabla["codigo"] = limpiar_codigo(tabla["codigo"])
    for anio in ANIOS:
        tabla[anio] = a_numero(tabla[anio])

    return tabla.dropna(subset=["2025"]).reset_index(drop=True)


def leer_socios(ruta):
    '''Lee un archivo de Trade Map con la lista de paises socios de un producto.

    Estos archivos traen una fila de navegacion de la pagina web antes del
    encabezado real: hay que saltarla con skiprows=1.
    '''
    tabla = pd.read_csv(ruta, skiprows=1)
    tabla = tabla.drop(columns=[tabla.columns[0]])          # columna de sub-productos, siempre vacia
    tabla = tabla.rename(columns={tabla.columns[0]: "pais"})

    for columna in tabla.columns[1:]:                        # todo lo demas es numerico
        tabla[columna] = a_numero(tabla[columna])

    return tabla

In [ ]:
# Una funcion, cuatro archivos
co_exp    = leer_canasta(ARCHIVOS["co_exp"])
co_imp    = leer_canasta(ARCHIVOS["co_imp"])
mundo_exp = leer_canasta(ARCHIVOS["mundo_exp"])
mundo_imp = leer_canasta(ARCHIVOS["mundo_imp"])

for nombre, tabla in [("Colombia exporta", co_exp), ("Colombia importa", co_imp),
                      ("Mundo exporta", mundo_exp), ("Mundo importa", mundo_imp)]:
    total = tabla.loc[tabla["codigo"] == "TOTAL", "2025"].iloc[0]
    print(f"{nombre:18s} {len(tabla):3d} filas   TOTAL 2025 = {total:>18,.0f} miles USD")

## 1.4. Control de calidad antes de calcular nada

Dos comprobaciones que valen su peso en oro. La primera: ¿los capítulos suman el total que reporta la propia fuente? Si no suman, cualquier participación que calculemos estará mal repartida.

La segunda es específica de este cuaderno: el Grubel-Lloyd y los índices de Vollrath del Cuaderno 4 necesitan que **cada capítulo tenga exportaciones e importaciones a la vez**, en Colombia y en el mundo. Si falta algún capítulo en alguno de los cuatro archivos, esas métricas quedan cojas justo ahí.

In [ ]:
print("CONTROL 1 - Los capitulos suman el TOTAL que reporta Trade Map?\n")
for nombre, tabla in [("Colombia exp", co_exp), ("Colombia imp", co_imp),
                      ("Mundo exp", mundo_exp), ("Mundo imp", mundo_imp)]:
    total_fila     = tabla.loc[tabla["codigo"] == "TOTAL", "2025"].iloc[0]
    suma_capitulos = tabla.loc[tabla["codigo"] != "TOTAL", "2025"].sum()
    desvio_pct     = (suma_capitulos - total_fila) / total_fila * 100
    print(f"  {nombre:13s} suma={suma_capitulos:>18,.0f}  TOTAL={total_fila:>18,.0f}  desvio={desvio_pct:+.5f} %")

print("\n\nCONTROL 2 - Cobertura de capitulos en los cuatro archivos\n")
capitulos = {nombre: set(t["codigo"]) - {"TOTAL"} for nombre, t in
             [("co_exp", co_exp), ("co_imp", co_imp), ("mundo_exp", mundo_exp), ("mundo_imp", mundo_imp)]}
for nombre, conjunto in capitulos.items():
    print(f"  {nombre:10s} {len(conjunto)} capitulos")

comunes = set.intersection(*capitulos.values())
print(f"\n  Capitulos presentes en los CUATRO archivos: {len(comunes)}")
print(f"  Exporta pero no importa: {sorted(capitulos['co_exp'] - capitulos['co_imp'])}")
print(f"  Importa pero no exporta: {sorted(capitulos['co_imp'] - capitulos['co_exp'])}")

Los cuatro archivos cubren exactamente los mismos 97 capítulos y los desvíos frente al total declarado son inferiores a 0,0001 %. Esto significa dos cosas prácticas: que el Grubel-Lloyd se podrá calcular en los 97 capítulos sin excepciones, y que en el Cuaderno 4 los índices de Vollrath —que usan logaritmos y por tanto **explotan si algún valor es cero**— van a poder calcularse completos.

No siempre pasa. En economías con matrices comerciales más dispersas es normal encontrar decenas de capítulos con cero en algún flujo, y ahí hay que decidir explícitamente qué hacer con ellos. Que aquí no ocurra es un resultado del control, no un supuesto.

## 1.5. Dos vistas de Trade Map que no dicen lo mismo

Trade Map permite descargar la misma consulta en dos presentaciones: la vista **"serie temporal"** (un año por columna) y la vista **"indicadores"** (un año más tasas de crecimiento, distancias y concentración). Uno esperaría que el valor de 2025 fuera idéntico en ambas.

No lo es.

In [ ]:
# La vista "indicadores" del mismo archivo de exportaciones colombianas
tabla_ind = max(pd.read_html(ARCHIVOS["co_exp_ind"]), key=lambda t: t.shape[0])
tabla_ind = tabla_ind.iloc[2:].copy()
tabla_ind.columns = ["codigo", "producto"] + [f"v{i}" for i in range(tabla_ind.shape[1] - 2)]
tabla_ind["codigo"]  = limpiar_codigo(tabla_ind["codigo"])
tabla_ind["x_indic"] = a_numero(tabla_ind["v0"])

comparacion = co_exp[["codigo", "producto", "2025"]].merge(
    tabla_ind[["codigo", "x_indic"]], on="codigo"
)
comparacion["diferencia"]     = comparacion["2025"] - comparacion["x_indic"]
comparacion["diferencia_pct"] = comparacion["diferencia"] / comparacion["x_indic"] * 100

difieren = comparacion[comparacion["diferencia"] != 0]
print(f"Capitulos en los que las dos vistas NO coinciden: {len(difieren)} de {len(comparacion)}\n")
print(difieren.reindex(difieren["diferencia"].abs().sort_values(ascending=False).index)
              .head(5)[["codigo", "2025", "x_indic", "diferencia", "diferencia_pct"]]
              .to_string(index=False))

**Decisión metodológica declarada.** Este cuaderno y el siguiente usan **siempre la vista "serie temporal"**, por dos razones: tiene los cinco años calculados con el mismo criterio, y por tanto las comparaciones año contra año son internamente consistentes. Mezclar las dos vistas —tomar 2021-2024 de una y 2025 de la otra— produce series con un salto artificial en el último año.

Es una decisión, no una verdad. Lo que no es opcional es **declararla**: cualquiera que replique este análisis con la otra vista obtendrá números ligeramente distintos, y debe poder saber por qué.

## 1.6. La cuarta fuente: los registros aduaneros de la DIAN

Hasta ahora todas las cifras venían de Trade Map, que es una base de **datos espejo**: reconstruye el comercio de un país combinando lo que ese país declara con lo que sus socios declaran. Legiscomex es otra cosa: entrega el **registro aduanero declaración por declaración**, tal como lo capturó la DIAN.

Para este caso se extrajeron 273.033 declaraciones de exportación de claveles frescos entre enero de 2020 y junio de 2026, ya agregadas por país de destino. Dos detalles de la extracción que conviene conocer:

- El código `0603120000` **no existe** en el arancel colombiano. El HS 060312 se abre aquí en dos subpartidas de diez dígitos, `0603121000` (claveles miniatura) y `0603129000` (los demás). Hay que sumar las dos.
- El módulo admite **máximo un año por consulta**, así que la serie son siete descargas consolidadas. Ninguna superó el tope de 80.000 registros, de modo que no hubo truncamiento.

In [ ]:
legis = pd.read_csv(ARCHIVOS["legis_destinos"])
auditoria = pd.read_csv(ARCHIVOS["legis_auditoria"])

print("Auditoria de las siete descargas:\n")
print(auditoria.to_string(index=False))

print("\n\nDestinos por anio (ya agregado):", legis.shape)
print(legis.head(4).to_string(index=False))

In [ ]:
# El anio 2026 esta incompleto: solo llega hasta junio.
# Lo separamos para no compararlo nunca contra anios completos.
ANIOS_COMPLETOS = sorted(a for a in legis["anio"].unique() if a != 2026)

legis_completo = legis[legis["anio"].isin(ANIOS_COMPLETOS)].copy()
legis_parcial  = legis[legis["anio"] == 2026].copy()

print(f"Anios completos que usaremos : {ANIOS_COMPLETOS}")
print(f"Anio parcial que apartamos   : 2026 (enero a junio)")
print(f"\nDestinos distintos en 2025   : {legis_completo[legis_completo['anio'] == 2025].shape[0]}")

**Para pensar.** Trade Map y Legiscomex miden el mismo comercio con métodos distintos, así que hay dos formas de que difieran y solo una es un problema.

Si difieren en el **total** de lo que Colombia exportó, alguien está midiendo mal. Si difieren en lo que Colombia le vendió a **un socio concreto**, puede ser simplemente que Colombia declara su exportación en valor **FOB** —lo que vale la mercancía puesta en el puerto de salida— mientras que el socio declara su importación en valor **CIF**, que incluye flete y seguro. Sobre un producto que viaja en avión, esa diferencia no es un detalle contable.

Vamos a encontrarnos con las dos situaciones en este cuaderno. Antes de seguir: ¿cuál de las dos te preocuparía más si tuvieras que firmar el informe?

---
# 2. Métrica 4 — Índice de Herfindahl-Hirschman (HHI)

## 2.1. Qué es

El HHI mide qué tan repartidas —o qué tan concentradas— están las exportaciones de un país entre sus distintos mercados de destino, o entre sus distintos productos. Es el mismo índice que usan las autoridades antimonopolio para medir concentración de mercado entre empresas; aquí se aplica a países o a productos en lugar de a empresas.

## 2.2. Para qué sirve y cómo se usa

El HHI es un **indicador de riesgo, no de rentabilidad**. Antes de celebrar que las exportaciones de un producto crecieron 30 % en un año, un analista serio pregunta: ¿ese crecimiento se apoya en muchos clientes o en uno solo?

Se calcula típicamente en dos direcciones:

- **HHI por destino** — qué tan repartidas están las ventas entre países compradores.
- **HHI por producto** — qué tan repartida está la canasta exportadora entre distintos bienes.

Ambas lecturas se usan en la práctica profesional para decidir si una estrategia de diversificación de mercados debe ser la prioridad número uno **antes** de invertir en aumentar volumen.

## 2.3. La fórmula

$$HHI = \sum_{i=1}^{n} s_i^2$$

## 2.4. Explicación matemática detallada

$s_i$ es la participación (*share*) del destino o producto $i$ en el total exportado, expresada como proporción decimal. Si Alemania compra el 30 % de las exportaciones de un producto, $s_i = 0{,}30$.

La fórmula eleva cada participación al cuadrado antes de sumarlas. Elevar al cuadrado no es un capricho: es lo que le da al índice su propiedad más importante, la **penalización desproporcionada de la concentración**.

Compárense dos escenarios con el mismo número de destinos:

- **Escenario A (diversificado):** cada uno de cuatro países compra el 25 %.
  $HHI_A = 0{,}25^2 \times 4 = 0{,}25$
- **Escenario B (concentrado):** un país compra el 70 % y los otros tres se reparten el 30 % restante.
  $HHI_B = 0{,}70^2 + 0{,}10^2 + 0{,}10^2 + 0{,}10^2 = 0{,}52$

Ambos tienen cuatro compradores, pero el HHI de B es más del doble. Pasar de 25 % a 70 % en un solo destino aporta $0{,}49 - 0{,}0625 = 0{,}4275$ puntos adicionales, mientras que repartir el resto entre tres países pequeños casi no compensa. Es exactamente el comportamiento que se quiere de un indicador de riesgo: **un solo comprador dominante debe encender más alarmas que varios compradores medianos.**

## 2.5. Ejemplo numérico paso a paso

In [ ]:
def hhi(valores):
    '''Indice de Herfindahl-Hirschman sobre una lista de valores absolutos.

    Convierte los valores en participaciones, las eleva al cuadrado y las suma.
    Devuelve un numero entre 0 (infinitos competidores iguales) y 1 (monopolio).
    '''
    valores = np.asarray(valores, dtype=float)
    valores = valores[~np.isnan(valores)]
    valores = valores[valores > 0]

    if valores.sum() == 0:
        return np.nan

    participaciones = valores / valores.sum()
    return float(np.sum(participaciones ** 2))


def numeros_equivalentes(indice_hhi):
    '''Traduce un HHI a "cuantos destinos del mismo tamano equivaldrian a esta reparticion".

    Es el inverso del HHI. Un HHI de 0.25 equivale a 4 destinos iguales;
    uno de 0.52, a menos de 2. Es la forma mas facil de comunicar el indice
    a alguien que no lo conoce.
    '''
    return 1 / indice_hhi

In [ ]:
# Reproducimos los dos escenarios del documento base
escenario_a = [25, 25, 25, 25]      # cuatro paises al 25 % cada uno
escenario_b = [70, 10, 10, 10]      # uno domina con 70 %

print("Los dos escenarios del documento base:\n")
for nombre, escenario in [("A (diversificado)", escenario_a), ("B (concentrado)", escenario_b)]:
    indice = hhi(escenario)
    print(f"  Escenario {nombre:20s} HHI = {indice:.4f}   ->  {numeros_equivalentes(indice):.2f} destinos equivalentes")

print(f"\n  Mismo numero de compradores (4), pero el escenario B es {hhi(escenario_b) / hhi(escenario_a):.2f} veces mas concentrado.")

El HHI del escenario B es 0,52 y su lectura en **números equivalentes** es 1,92: aunque hay cuatro compradores en la mesa, el riesgo que soporta la empresa es el de tener menos de dos.

Esa traducción es la que hace útil el índice en una reunión. Decir *"nuestro HHI es 0,52"* no significa nada para un comité directivo; decir *"tenemos cuatro clientes pero dependemos como si tuviéramos dos"* se entiende de inmediato.

## 2.6. Cálculo sobre nuestros datos reales — el HHI por destino de Colombia

Aquí es donde el dato aduanero paga su precio. Trade Map publica un índice de concentración ya calculado, pero solo para el último año y sin mostrar cómo lo obtuvo. Con las declaraciones de la DIAN podemos calcularlo nosotros, año por año, y ver la **trayectoria**.

In [ ]:
serie_hhi = []
for anio in ANIOS_COMPLETOS:
    del_anio = legis_completo[legis_completo["anio"] == anio]
    indice   = hhi(del_anio["valor_fob_usd"])
    mayor    = del_anio.nlargest(1, "valor_fob_usd").iloc[0]
    serie_hhi.append({
        "anio": anio,
        "destinos": len(del_anio),
        "hhi": indice,
        "equivalentes": numeros_equivalentes(indice),
        "mayor_destino": mayor["pais_destino"],
        "cuota_mayor_pct": mayor["valor_fob_usd"] / del_anio["valor_fob_usd"].sum() * 100,
        "fob_total_musd": del_anio["valor_fob_usd"].sum() / 1e6,
    })

serie_hhi = pd.DataFrame(serie_hhi)
print(serie_hhi.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))

ax.plot(serie_hhi["anio"], serie_hhi["hhi"], color=AZUL, linewidth=2.4,
        marker="o", markersize=7, zorder=3)

# Etiqueta directa sobre cada punto: el lector no deberia tener que ir al eje
for _, fila in serie_hhi.iterrows():
    ax.annotate(f"{fila['hhi']:.4f}", (fila["anio"], fila["hhi"]),
                textcoords="offset points", xytext=(0, 11),
                ha="center", fontsize=9, color=GRIS_TEXT)

ax.set_ylabel("HHI por destino", fontsize=10, color=GRIS_TEXT)
ax.set_title("Colombia diversifica sus destinos de clavel, muy despacio\n"
             "HHI de las exportaciones de HS 060312 por pais de destino, 2020-2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_ylim(0.19, 0.24)
ax.set_xticks(serie_hhi["anio"])
ax.yaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.06,
            "Un HHI que baja significa exportaciones mas repartidas entre destinos.\n"
            "Fuente: elaboracion propia con registros aduaneros de la DIAN via Legiscomex (2026).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

El índice baja de 0,2282 a 0,2094 en seis años. Traducido a números equivalentes: Colombia pasó de depender de **4,4 destinos** a depender de **4,8**. En seis años ganó menos de medio comprador equivalente.

Es una diversificación real, pero tan lenta que difícilmente resiste una lectura optimista. Y el detalle que la trayectoria no oculta: el primer destino se mantuvo en torno al 41 % durante todo el periodo. Colombia no está reduciendo su dependencia de Estados Unidos; está añadiendo destinos pequeños alrededor de una dependencia que no se mueve.

## 2.7. La validación cruzada: dos fuentes independientes, un mismo número

Trade Map reporta su propio índice de concentración para las exportaciones colombianas de este producto. Lo calculó con datos espejo. Nosotros lo calculamos con registros aduaneros. Si ambos coinciden, cada uno respalda al otro.

In [ ]:
tm_exp = pd.read_csv(ARCHIVOS["tm_exp_060312"], dtype={"reporterCd": str})
fila_colombia = tm_exp[tm_exp["reporterLabel"] == "Colombia"].iloc[0]

hhi_trademap   = float(fila_colombia["Concentration"])
hhi_legiscomex = serie_hhi.loc[serie_hhi["anio"] == 2025, "hhi"].iloc[0]

print("HHI por destino de las exportaciones colombianas de HS 060312, 2025\n")
print(f"  Trade Map (dato espejo, indice publicado) : {hhi_trademap:.4f}   -> {numeros_equivalentes(hhi_trademap):.2f} destinos equivalentes")
print(f"  Legiscomex / DIAN (calculado por nosotros): {hhi_legiscomex:.4f}   -> {numeros_equivalentes(hhi_legiscomex):.2f} destinos equivalentes")
print(f"\n  Diferencia absoluta : {abs(hhi_trademap - hhi_legiscomex):.4f}")
print(f"  Diferencia relativa : {abs(hhi_trademap - hhi_legiscomex) / hhi_trademap * 100:.2f} %")

# Y el total exportado, que es la comprobacion mas exigente
fob_legiscomex = legis_completo.loc[legis_completo["anio"] == 2025, "valor_fob_usd"].sum()
valor_trademap = float(str(fila_colombia["Value (kUSD)"]).replace(",", "")) * 1_000

print(f"\n  Total exportado 2025, Trade Map  : {valor_trademap:>16,.0f} USD")
print(f"  Total exportado 2025, DIAN       : {fob_legiscomex:>16,.0f} USD")
print(f"  Diferencia                       : {abs(valor_trademap - fob_legiscomex):>16,.0f} USD")

Los dos índices difieren en 0,0009, un 0,4 %. Y el total exportado coincide prácticamente al dólar.

Esto no es una casualidad afortunada: es la razón por la que Trade Map es confiable para el total de un país reportante. Su cifra colombiana **es**, en el fondo, la cifra de la DIAN. Lo que aporta el registro aduanero no es una corrección, sino algo que Trade Map no publica: la serie histórica del índice y la posibilidad de auditarlo declaración por declaración.

**Cuándo sí van a diferir.** Cuando comparemos lo que Colombia dice que vendió a un socio contra lo que ese socio dice que compró. Ahí entra el FOB contra el CIF, y lo veremos en la sección 5 con un caso que cambia la conclusión del curso.

## 2.8. El otro lado del índice: ¿qué tan atrincherado está el proveedor de Corea?

El Cuaderno 2 dejó una pregunta abierta: *"Sabemos que Corea importa, no de quién. ¿Quién es el proveedor a desplazar y qué tan atrincherado está?"*. El mismo HHI, aplicado del lado del comprador, la responde.

In [ ]:
proveedores = leer_socios(ARCHIVOS["kor_prov_serie"])

mundo_kor = proveedores.loc[proveedores["pais"] == "World", "2025"].iloc[0]
sin_mundo = proveedores[proveedores["pais"] != "World"].copy()
sin_mundo["cuota_pct"] = sin_mundo["2025"] / mundo_kor * 100
sin_mundo = sin_mundo.sort_values("2025", ascending=False)

print("Quien le vende claveles frescos a Corea del Sur (2025, miles USD CIF)\n")
print(sin_mundo[["pais", "2025", "cuota_pct"]].to_string(index=False))

hhi_proveedores = hhi(sin_mundo["2025"])
print(f"\n  HHI de proveedores : {hhi_proveedores:.4f}")
print(f"  Equivale a         : {numeros_equivalentes(hhi_proveedores):.2f} proveedores del mismo tamano")

In [ ]:
# Comprobamos contra el indice que publica Trade Map para el mercado coreano
tm_imp = pd.read_csv(ARCHIVOS["tm_imp_060312"], dtype={"reporterCd": str})
fila_corea = tm_imp[tm_imp["reporterLabel"].str.contains("Corea|Korea", case=False, na=False)].iloc[0]

print(f"  Nuestro calculo      : {hhi_proveedores:.4f}")
print(f"  Trade Map publica    : {float(fila_corea['Concentration']):.4f}")
print(f"  Diferencia           : {abs(hhi_proveedores - float(fila_corea['Concentration'])):.4f}")

Coincidencia exacta hasta el cuarto decimal. Y el resultado responde la pregunta del Cuaderno 2 de una forma que nadie esperaba: **el proveedor atrincherado en Corea del Sur es Colombia.**

Con un 86,65 % del mercado coreano, no hay a quién desplazar. La pregunta de negocio, entonces, deja de ser *"¿cómo entramos?"* y pasa a ser otra completamente distinta, que retomaremos en la sección 5.

In [ ]:
datos = sin_mundo.head(6).iloc[::-1]
colores = [NARANJA if p == "Colombia" else AZUL for p in datos["pais"]]

fig, ax = plt.subplots(figsize=(9, 4.2))
barras = ax.barh(datos["pais"], datos["cuota_pct"], color=colores, height=0.66)

for barra, valor in zip(barras, datos["cuota_pct"]):
    ax.text(valor + 1.4, barra.get_y() + barra.get_height() / 2,
            f"{valor:.2f} %", va="center", fontsize=9.5, color=GRIS_TEXT)

ax.set_xlabel("Cuota del mercado coreano de claveles frescos (%)", fontsize=10, color=GRIS_TEXT)
ax.set_title("El proveedor a desplazar en Corea del Sur es Colombia\n"
             "Proveedores de HS 060312 a Corea del Sur, 2025 (HHI = 0,77)",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_xlim(0, 100)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.05,
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025), declarados por Corea del Sur.",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 2.9. La segunda dirección del índice: el HHI por producto

Hasta aquí medimos concentración de **clientes**. La otra pregunta, igual de importante, es la concentración de **productos**: de todo lo que Colombia le vende al mundo, ¿cuánto depende de unos pocos bienes?

Aquí ya no hablamos de claveles sino de la economía entera, con los 97 capítulos del Sistema Armonizado.

In [ ]:
canasta = co_exp[co_exp["codigo"] != "TOTAL"].copy()
canasta["participacion"] = canasta["2025"] / canasta["2025"].sum()

hhi_canasta = hhi(canasta["2025"])

print(f"HHI de la canasta exportadora colombiana (97 capitulos HS, 2025)\n")
print(f"  HHI          : {hhi_canasta:.4f}")
print(f"  Equivale a   : {numeros_equivalentes(hhi_canasta):.1f} capitulos del mismo tamano")
print(f"\nLos seis capitulos que mas pesan:\n")

top = canasta.nlargest(6, "2025")[["codigo", "producto", "2025", "participacion"]].copy()
top["participacion"] = (top["participacion"] * 100).round(2)
top["producto"] = top["producto"].str[:52]
print(top.to_string(index=False))
print(f"\n  Los seis juntos: {canasta.nlargest(6, '2025')['participacion'].sum() * 100:.1f} % de todo lo que Colombia exporta.")

Un HHI de 0,15 sobre 97 capítulos posibles equivale a exportar **6,5 productos** del mismo tamaño. Colombia tiene una economía formalmente diversificada —vende algo en 97 de los 97 capítulos del arancel— pero concentrada de hecho en un puñado de ellos.

Fíjate en el contraste con el HHI por destino: 0,21 para los destinos del clavel, 0,15 para la canasta entera. El segundo número es más bajo, y sin embargo es el más preocupante de los dos, porque un producto se sustituye mucho más despacio que un cliente.

## 2.10. Una advertencia sobre escalas

El HHI se reporta a veces en escala 0–1 (como aquí, sumando proporciones al cuadrado) y otras veces en escala 0–10.000 (sumando porcentajes al cuadrado, convención heredada de la práctica antimonopolio de Estados Unidos).

In [ ]:
print("El mismo indice, en las dos escalas que circulan en la literatura:\n")
for etiqueta, indice in [("Destinos del clavel (Colombia)", hhi_legiscomex),
                         ("Proveedores de Corea del Sur", hhi_proveedores),
                         ("Canasta exportadora colombiana", hhi_canasta)]:
    print(f"  {etiqueta:34s} {indice:.4f}  (escala 0-1)  =  {indice * 10_000:>8,.0f}  (escala 0-10.000)")

print("\nUmbrales antimonopolio de Estados Unidos, en la escala de porcentajes:")
print("  por debajo de 1.500  ->  mercado no concentrado")
print("  entre 1.500 y 2.500  ->  moderadamente concentrado")
print("  por encima de 2.500  ->  altamente concentrado")

**Antes de comparar el HHI reportado en dos estudios distintos, siempre hay que verificar en qué escala está expresado cada uno.** Un HHI de 0,38 y uno de 3.800 son el mismo número de concentración.

## 2.11. Interpretación y umbrales

| HHI (escala 0–1) | Números equivalentes | Lectura |
|---|---|---|
| menor que 0,15 | más de 6,7 | Repartido |
| 0,15 a 0,25 | 4 a 6,7 | Moderadamente concentrado |
| mayor que 0,25 | menos de 4 | Altamente concentrado |

Los tres índices que calculamos caen en tres casillas distintas, y esa es justamente la utilidad del ejercicio.

## 2.12. Impacto en el negocio y la decisión que habilita

Para la empresa del caso, el HHI convierte una intuición en un número defendible ante un comité: sus destinos de clavel equivalen a menos de cinco clientes, y esa cifra apenas se ha movido en seis años. Eso ordena la prioridad: **diversificar mercados antes que aumentar volumen**, porque crecer 30 % sobre una base de 4,8 destinos equivalentes multiplica el riesgo en lugar de repartirlo.

Para el lado del comprador, el HHI de proveedores es una herramienta de negociación. Un mercado con HHI de 0,77 y un solo proveedor dominante es un mercado donde el comprador tiene incentivos activos para buscar alternativas — y donde el proveedor dominante tiene mucho que perder.

## 2.13. De dónde se saca exactamente el dato

La matriz de participaciones $s_i$ se construye con los valores de exportación por producto (HS) o por país de destino de **Trade Map** (ITC, 2025), o con los registros aduaneros de la **DIAN** vía Legiscomex cuando se necesita la serie histórica o trazabilidad declaración por declaración.

Estudios recientes confirman el cálculo directo sobre Trade Map como estándar: salmón (García Juárez et al., 2026), maíz (Arbulú Ballesteros et al., 2026), banano andino (Corrales Otazú et al., 2025) y café (García Juárez et al., 2025).

---
# 3. Métrica 5 — Índice de Theil

## 3.1. Qué es

Derivado de la teoría de la entropía de la información —la misma matemática que mide "desorden" o "sorpresa" en teoría de la información—, el índice de Theil mide concentración igual que el HHI, pero con una capacidad adicional: **puede descomponerse** en un componente "entre grupos" y un componente "dentro de los grupos".

## 3.2. Para qué sirve y cómo se usa

Se usa cuando la pregunta de negocio no es solo *"¿qué tan concentrado estoy?"* sino *"¿la diversificación que estoy logrando es real o es solo apariencia?"*.

Un país que hace veinte años exportaba solo café y hoy exporta café, flores y banano parece diversificado si se mira superficialmente. Pero si los tres productos pertenecen al mismo macro-sector agroindustrial primario y ninguno es manufactura o servicios, la diversificación *dentro del grupo agroindustrial* puede ser alta mientras la diversificación *entre grupos* sigue siendo prácticamente nula.

El índice de Theil separa exactamente esa señal. **El HHI no puede hacerlo por sí solo**, y esa es la única razón por la que se calculan los dos.

## 3.3. La fórmula

$$T = \frac{1}{n}\sum_{i=1}^{n} \frac{x_i}{\mu} \ln\!\left(\frac{x_i}{\mu}\right)$$

donde $x_i$ es el valor exportado del producto $i$, $\mu$ es el promedio de todos los $x_i$ y $n$ el número de productos.

Y su descomposición, que es lo que realmente importa:

$$T = \underbrace{\sum_{g} s_g \ln\!\left(\frac{s_g}{n_g/n}\right)}_{\text{entre grupos}} + \underbrace{\sum_{g} s_g \, T_g}_{\text{dentro de los grupos}}$$

donde $s_g$ es la participación del grupo $g$ en el total exportado, $n_g$ el número de productos del grupo y $T_g$ el índice de Theil calculado solo dentro de ese grupo.

## 3.4. Explicación matemática detallada

La descomposición se apoya en la **propiedad aditiva de la entropía**: la dispersión total de una canasta exportadora puede escribirse como la suma de dos piezas que no se solapan.

El primer término compara, para cada grupo, **el peso económico que tiene** ($s_g$) contra **el peso que le correspondería por número de productos** ($n_g/n$). Si un grupo con 3 de 96 capítulos concentra el 40 % de las exportaciones, ese cociente es grande, su logaritmo es positivo, y el término suma. Si todos los grupos pesaran exactamente en proporción a cuántos capítulos tienen, cada logaritmo valdría $\ln(1) = 0$ y el componente "entre grupos" sería cero: la concentración estaría toda **dentro** de los grupos.

El segundo término es el promedio ponderado de la concentración interna de cada grupo. Mide si, dentro del sector agroindustrial, todo depende del café o está repartido entre café, flores, banano y frutas.

La suma de ambos da **exactamente** el índice total. No es una aproximación, y esa exactitud nos sirve como control de calidad: si al calcularlos por separado no suman el total, hay un error en el código.

## 3.5. Los grupos: las 21 secciones del Sistema Armonizado

Para descomponer hace falta definir los grupos. El propio Sistema Armonizado los trae: sus 97 capítulos se agrupan en **21 secciones** que corresponden a macro-sectores económicos. Esta correspondencia no se descarga de ninguna base: es parte de la nomenclatura y se codifica a mano.

In [ ]:
# Las 21 secciones del Sistema Armonizado: que capitulos agrupa cada una
SECCIONES_HS = {
    "I":     (range(1, 6),   "Animales vivos y productos del reino animal"),
    "II":    (range(6, 15),  "Productos del reino vegetal"),
    "III":   (range(15, 16), "Grasas y aceites"),
    "IV":    (range(16, 25), "Alimentos, bebidas y tabaco"),
    "V":     (range(25, 28), "Productos minerales"),
    "VI":    (range(28, 39), "Industrias quimicas"),
    "VII":   (range(39, 41), "Plastico y caucho"),
    "VIII":  (range(41, 44), "Pieles y cueros"),
    "IX":    (range(44, 47), "Madera y corcho"),
    "X":     (range(47, 50), "Pasta de madera y papel"),
    "XI":    (range(50, 64), "Materias textiles"),
    "XII":   (range(64, 68), "Calzado y sombrereria"),
    "XIII":  (range(68, 71), "Piedra, cemento, ceramica y vidrio"),
    "XIV":   (range(71, 72), "Perlas, piedras y metales preciosos"),
    "XV":    (range(72, 84), "Metales comunes"),
    "XVI":   (range(84, 86), "Maquinas y material electrico"),
    "XVII":  (range(86, 90), "Material de transporte"),
    "XVIII": (range(90, 93), "Optica, precision y relojeria"),
    "XIX":   (range(93, 94), "Armas y municiones"),
    "XX":    (range(94, 97), "Mercancias y productos diversos"),
    "XXI":   (range(97, 98), "Objetos de arte y antiguedades"),
}

# Le damos la vuelta: de "seccion -> capitulos" a "capitulo -> seccion"
CAPITULO_A_SECCION = {
    f"{capitulo:02d}": seccion
    for seccion, (capitulos, _) in SECCIONES_HS.items()
    for capitulo in capitulos
}
NOMBRE_SECCION = {seccion: nombre for seccion, (_, nombre) in SECCIONES_HS.items()}

print(f"Secciones definidas : {len(SECCIONES_HS)}")
print(f"Capitulos mapeados  : {len(CAPITULO_A_SECCION)}")
print(f"\nEjemplos: HS06 -> seccion {CAPITULO_A_SECCION['06']} ({NOMBRE_SECCION[CAPITULO_A_SECCION['06']]})")
print(f"          HS27 -> seccion {CAPITULO_A_SECCION['27']} ({NOMBRE_SECCION[CAPITULO_A_SECCION['27']]})")
print(f"          HS85 -> seccion {CAPITULO_A_SECCION['85']} ({NOMBRE_SECCION[CAPITULO_A_SECCION['85']]})")

In [ ]:
canasta["seccion"] = canasta["codigo"].map(CAPITULO_A_SECCION)

sin_seccion = canasta[canasta["seccion"].isna()]
print("Capitulos que no pertenecen a ninguna seccion del Sistema Armonizado:\n")
print(sin_seccion[["codigo", "producto", "2025"]].to_string(index=False))
print(f"\nPeso en la canasta: {sin_seccion['2025'].sum() / canasta['2025'].sum() * 100:.3f} %")

Aparece el capítulo 99, *"Commodities not elsewhere specified"*, que no es un sector sino el cajón de lo no clasificado. No pertenece a ninguna sección porque no es una categoría económica.

**Decisión declarada:** lo excluimos de la descomposición de Theil. Pesa un 0,02 % de la canasta, así que su exclusión no mueve el resultado, pero incluirlo obligaría a inventar un grupo que no existe en la nomenclatura y rompería la exactitud aritmética de la descomposición. Se declara aquí y se documenta en la ficha metodológica del final.

In [ ]:
def theil(valores):
    '''Indice de Theil sobre una lista de valores positivos.

    Mide dispersion usando entropia: cuanto mas concentrado el reparto,
    mayor el indice. A diferencia del HHI, no tiene techo en 1.
    '''
    valores = np.asarray(valores, dtype=float)
    valores = valores[valores > 0]
    n = len(valores)
    media = valores.mean()

    return float((1 / n) * np.sum((valores / media) * np.log(valores / media)))


def theil_descompuesto(tabla, columna_valor, columna_grupo):
    '''Descompone el indice de Theil en sus componentes entre y dentro de grupos.

    Devuelve un diccionario con el total, los dos componentes y el residuo,
    que debe ser cero: si no lo es, hay un error en el calculo.
    '''
    tabla = tabla[tabla[columna_valor] > 0].copy()

    total_exportado = tabla[columna_valor].sum()
    n_productos     = len(tabla)

    entre  = 0.0
    dentro = 0.0

    for _, grupo in tabla.groupby(columna_grupo):
        peso_grupo     = grupo[columna_valor].sum() / total_exportado    # s_g
        peso_por_conteo = len(grupo) / n_productos                       # n_g / n

        entre  += peso_grupo * np.log(peso_grupo / peso_por_conteo)
        dentro += peso_grupo * theil(grupo[columna_valor])

    total = theil(tabla[columna_valor])

    return {
        "total": total,
        "entre_grupos": entre,
        "dentro_grupos": dentro,
        "residuo": total - (entre + dentro),
        "pct_entre": entre / total * 100,
        "pct_dentro": dentro / total * 100,
    }

In [ ]:
canasta_theil = canasta.dropna(subset=["seccion"]).copy()
resultado = theil_descompuesto(canasta_theil, "2025", "seccion")

print(f"Indice de Theil de la canasta exportadora colombiana, 2025")
print(f"({len(canasta_theil)} capitulos agrupados en {canasta_theil['seccion'].nunique()} secciones HS)\n")
print(f"  Theil total          : {resultado['total']:.6f}")
print(f"    entre grupos       : {resultado['entre_grupos']:.6f}   ({resultado['pct_entre']:.1f} %)")
print(f"    dentro de grupos   : {resultado['dentro_grupos']:.6f}   ({resultado['pct_dentro']:.1f} %)")
print(f"\n  CONTROL - residuo    : {resultado['residuo']:.2e}   (debe ser cero)")

assert abs(resultado["residuo"]) < 1e-9, "La descomposicion no cierra: revisa el calculo"
print("\n  La descomposicion cierra exactamente.")

In [ ]:
# Concentracion dentro de cada seccion, para ver donde esta el "dentro de grupos"
por_seccion = canasta_theil.groupby("seccion").agg(
    capitulos=("codigo", "count"),
    valor_musd=("2025", lambda x: x.sum() / 1000),
).reset_index()
por_seccion["nombre"] = por_seccion["seccion"].map(NOMBRE_SECCION)
por_seccion["peso_pct"] = por_seccion["valor_musd"] / por_seccion["valor_musd"].sum() * 100
por_seccion["theil_interno"] = [
    theil(canasta_theil.loc[canasta_theil["seccion"] == s, "2025"])
    for s in por_seccion["seccion"]
]
por_seccion = por_seccion.sort_values("peso_pct", ascending=False)

print("Las secciones que mas pesan en la canasta colombiana:\n")
print(por_seccion.head(8)[["seccion", "nombre", "capitulos", "peso_pct", "theil_interno"]]
                 .to_string(index=False))

In [ ]:
datos = por_seccion.head(8).iloc[::-1]

fig, ax = plt.subplots(figsize=(9.5, 5))
barras = ax.barh(datos["nombre"].str[:34], datos["peso_pct"], color=AZUL, height=0.66)

for barra, peso, theil_int in zip(barras, datos["peso_pct"], datos["theil_interno"]):
    ax.text(peso + 0.7, barra.get_y() + barra.get_height() / 2,
            f"{peso:.1f} %   (Theil interno {theil_int:.2f})",
            va="center", fontsize=9, color=GRIS_TEXT)

ax.set_xlabel("Participación en las exportaciones colombianas (%)", fontsize=10, color=GRIS_TEXT)
ax.set_title("El 62 % de la concentración colombiana está ENTRE sectores, no dentro\n"
             "Secciones del Sistema Armonizado por peso exportador, 2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_xlim(0, datos["peso_pct"].max() * 1.55)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.05,
            "El Theil interno mide la concentracion DENTRO de cada seccion.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 3.6. Interpretación y umbrales

Un índice de Theil total alto, con la mayor parte de su valor explicado por el componente **entre grupos**, indica diversificación estructural genuina o, leído al revés, concentración estructural. Un Theil alto explicado mayormente por el componente **dentro de los grupos** indica que el país sigue dependiendo del mismo macro-sector, aunque con más variedad de productos individuales dentro de él.

En Colombia, el 61,7 % del índice viene del componente **entre secciones**. La lectura es incómoda pero clara: la concentración exportadora colombiana no es un problema de tener pocos productos dentro de sus sectores, sino de que **unos pocos sectores absorben casi todo**. Productos minerales y productos del reino vegetal, dos secciones de 3 y 9 capítulos sobre 96, explican por sí solas más de la mitad de lo que Colombia le vende al mundo.

Es exactamente la señal que el HHI no podía dar. El HHI decía "equivales a 6,5 productos"; el Theil dice **dónde** está esa concentración y, por tanto, dónde tendría que actuar una política de diversificación.

## 3.7. Impacto en el negocio y la decisión que habilita

Para un ministerio de comercio o una agencia de promoción de exportaciones, la distinción cambia por completo el diseño de política: si la diversificación fuera solo "dentro del grupo", los incentivos deberían dirigirse a sectores completamente nuevos. Como en Colombia el problema es "entre grupos", la conclusión es la misma pero más urgente — y además señala que añadir un producto agroindustrial más apenas movería el índice.

Para un inversionista de fondos soberanos o de *private equity*, un país con componente "entre grupos" decreciente en el tiempo es señal de transformación productiva real y, por tanto, de menor riesgo estructural de largo plazo.

Zhang et al. (2023) y Chang et al. (2024) aplican esta lógica a canastas de productos creativos. Trinh y Nguyen (2021) van más allá y muestran una relación **no lineal** entre diversificación y crecimiento: por debajo de cierto umbral de diversificación estructural, el efecto de diversificar sobre el crecimiento del PIB no es estadísticamente significativo. Diversificar "un poco" dentro del mismo sector no genera el beneficio macroeconómico que sí genera diversificar hacia sectores nuevos.

## 3.8. De dónde se saca exactamente el dato

Igual que el HHI: la matriz de exportaciones por producto de Trade Map, agregada según la clasificación jerárquica (2, 4 o 6 dígitos HS) que se quiera usar para la descomposición. La correspondencia capítulo → sección es parte de la nomenclatura del Sistema Armonizado, no de la base de datos, y se codifica manualmente como hicimos arriba.

---
# 4. Métrica 6 — Índice de Grubel-Lloyd

## 4.1. Qué es

El índice de Grubel-Lloyd mide si el comercio de un país en una industria específica es **interindustrial** —el país solo exporta o solo importa ese tipo de bien, nunca ambos— o **intraindustrial** —el país exporta y importa simultáneamente bienes muy similares dentro de la misma industria—.

Alemania exporta e importa componentes automotrices al mismo tiempo: eso es comercio intraindustrial. Colombia exporta café y no importa café: eso es interindustrial puro.

## 4.2. Para qué sirve y cómo se usa

Se usa para diagnosticar la **naturaleza** de la inserción internacional de un país en una industria, más allá de si tiene superávit o déficit.

Las teorías clásicas del comercio (Ricardo, Heckscher-Ohlin) predicen comercio interindustrial, basado en diferencias de dotación de factores. Pero el comercio real del siglo XXI ocurre mayoritariamente entre economías que intercambian bienes de la misma industria, producidos en distintas etapas de una cadena global de valor.

Un país con GL alto en manufactura está señalando que sus empresas participan en cadenas de producción compartidas con sus socios: reciben insumos, agregan valor y reexportan, en lugar de operar como productor autosuficiente de principio a fin.

## 4.3. La fórmula

$$GL_i = 1 - \frac{|X_i - M_i|}{X_i + M_i}$$

## 4.4. Explicación matemática detallada

Obsérvese el parecido con el **IBCR del Cuaderno 2**: el término $\frac{|X_i - M_i|}{X_i + M_i}$ es exactamente el valor absoluto del IBCR.

Restar ese término de 1 invierte la lógica. El IBCR mide qué tan **desbalanceado** está el comercio: cerca de ±1 significa comercio unidireccional. El GL mide qué tan **balanceado** está: cerca de 1 significa que $X_i$ y $M_i$ son casi iguales en magnitud, es decir, comercio bidireccional intenso dentro de la misma industria.

Los dos casos extremos:

- Si $X_i = M_i$ exactamente, el numerador del término restado es cero y $GL_i = 1 - 0 = 1$: comercio perfectamente intraindustrial.
- Si el país solo exporta o solo importa, $|X_i - M_i| = X_i + M_i$, el término restado vale 1 y $GL_i = 1 - 1 = 0$: comercio puramente interindustrial.

Esta relación con el IBCR no es una curiosidad: significa que **ya calculaste el Grubel-Lloyd en el Cuaderno 2 sin saberlo**. Vamos a demostrarlo en código.

## 4.5. Ejemplo numérico paso a paso

In [ ]:
def grubel_lloyd(exportaciones, importaciones):
    '''Indice de Grubel-Lloyd: mide comercio intraindustrial.

    Devuelve 1 cuando el pais exporta e importa exactamente lo mismo del bien
    (comercio intraindustrial puro) y 0 cuando solo hace una de las dos cosas
    (comercio interindustrial puro).
    '''
    exportaciones = np.asarray(exportaciones, dtype=float)
    importaciones = np.asarray(importaciones, dtype=float)
    comercio_total = exportaciones + importaciones

    return np.where(
        comercio_total > 0,
        1 - np.abs(exportaciones - importaciones) / comercio_total,
        np.nan,
    )


def ibcr(exportaciones, importaciones):
    '''Indice de Balanza Comercial Relativa, el mismo del Cuaderno 2.'''
    exportaciones = np.asarray(exportaciones, dtype=float)
    importaciones = np.asarray(importaciones, dtype=float)
    comercio_total = exportaciones + importaciones

    return np.where(comercio_total > 0,
                    (exportaciones - importaciones) / comercio_total,
                    np.nan)

In [ ]:
print("Los dos ejemplos del documento base:\n")

# Un pais exporta 500 millones de autopartes e importa 420 (misma partida HS 8708)
gl_autopartes = grubel_lloyd(500, 420)
print(f"  Autopartes  X=500  M=420  ->  GL = {gl_autopartes:.3f}")
print( "              comercio casi enteramente intraindustrial: el pais esta")
print( "              integrado en una cadena regional de produccion automotriz.\n")

# Un pais exporta 300 millones de cafe verde y no importa nada de cafe
gl_cafe = grubel_lloyd(300, 0)
print(f"  Cafe verde  X=300  M=0    ->  GL = {gl_cafe:.3f}")
print( "              comercio puramente interindustrial: productor primario neto,")
print( "              sin integracion en una cadena donde tambien compre el bien.")

In [ ]:
# Demostracion de que GL = 1 - |IBCR|, la relacion con el Cuaderno 2
print("GL = 1 - |IBCR|  ->  comprobacion sobre cinco casos:\n")
print(f"{'X':>8} {'M':>8} {'IBCR':>9} {'1-|IBCR|':>10} {'GL':>9}  coinciden")
print("-" * 58)
for x, m in [(500, 420), (300, 0), (0, 300), (100, 100), (900, 100)]:
    valor_ibcr = float(ibcr(x, m))
    valor_gl   = float(grubel_lloyd(x, m))
    print(f"{x:>8} {m:>8} {valor_ibcr:>+9.3f} {1 - abs(valor_ibcr):>10.3f} {valor_gl:>9.3f}"
          f"       {'si' if abs((1 - abs(valor_ibcr)) - valor_gl) < 1e-12 else 'NO'}")

## 4.6. Cálculo sobre nuestros datos reales

Ahora sobre los 97 capítulos de la economía colombiana, cruzando exportaciones e importaciones del mismo código HS.

In [ ]:
gl_tabla = (
    co_exp[co_exp["codigo"] != "TOTAL"][["codigo", "producto", "2025"]]
        .rename(columns={"2025": "exportado"})
        .merge(
            co_imp[co_imp["codigo"] != "TOTAL"][["codigo", "2025"]]
                .rename(columns={"2025": "importado"}),
            on="codigo", how="inner",
        )
)

gl_tabla["gl"]      = grubel_lloyd(gl_tabla["exportado"], gl_tabla["importado"])
gl_tabla["ibcr"]    = ibcr(gl_tabla["exportado"], gl_tabla["importado"])
gl_tabla["comercio"] = gl_tabla["exportado"] + gl_tabla["importado"]
gl_tabla["seccion"] = gl_tabla["codigo"].map(CAPITULO_A_SECCION)

gl_ponderado = np.average(gl_tabla["gl"], weights=gl_tabla["comercio"])

print(f"Capitulos con exportaciones e importaciones: {gl_tabla['gl'].notna().sum()} de {len(gl_tabla)}")
print(f"Grubel-Lloyd promedio ponderado por comercio: {gl_ponderado:.4f}\n")
print("El promedio se pondera por el comercio total del capitulo: un capitulo")
print("marginal no puede pesar lo mismo que uno que mueve miles de millones.")

In [ ]:
print("Los cinco capitulos MAS intraindustriales de Colombia:\n")
print(gl_tabla.nlargest(5, "gl")[["codigo", "producto", "exportado", "importado", "gl"]]
              .assign(producto=lambda d: d["producto"].str[:44]).to_string(index=False))

print("\n\nLos cinco MENOS intraindustriales, entre los capitulos que mueven mas comercio:\n")
grandes = gl_tabla.nlargest(30, "comercio")
print(grandes.nsmallest(5, "gl")[["codigo", "producto", "exportado", "importado", "gl"]]
             .assign(producto=lambda d: d["producto"].str[:44]).to_string(index=False))

print("\n\nEl capitulo del caso:\n")
print(gl_tabla[gl_tabla["codigo"] == "06"][["codigo", "producto", "exportado", "importado", "gl", "ibcr"]]
      .to_string(index=False))

In [ ]:
datos = gl_tabla.nlargest(14, "comercio").sort_values("gl")
colores = [NARANJA if c == "06" else AZUL for c in datos["codigo"]]
etiquetas = [f"HS{c} · {p[:26]}" for c, p in zip(datos["codigo"], datos["producto"])]

fig, ax = plt.subplots(figsize=(9.5, 6))
barras = ax.barh(etiquetas, datos["gl"], color=colores, height=0.66)

for barra, valor in zip(barras, datos["gl"]):
    ax.text(valor + 0.016, barra.get_y() + barra.get_height() / 2,
            f"{valor:.3f}", va="center", fontsize=9, color=GRIS_TEXT)

ax.axvline(0.45, color=GRIS_MID, linewidth=1.3, linestyle="--", zorder=1)
ax.text(0.455, -0.85, "0,45 = umbral de base manufacturera interconectada",
        fontsize=8.5, color=GRIS_EJE)

ax.set_xlabel("Índice de Grubel-Lloyd   (0 = interindustrial puro   |   1 = intraindustrial puro)",
              fontsize=10, color=GRIS_TEXT)
ax.set_title("Colombia comercia flores como productor primario, no como eslabón de cadena\n"
             "Grubel-Lloyd de los 14 capítulos con más comercio, 2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_xlim(0, 1.05)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=8.5)

plt.figtext(0.01, -0.04,
            "En naranja: HS06, el capitulo de las flores.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 4.7. Interpretación y umbrales

En los reportes de política comercial, cruzar la frontera de un GL de **0,45** hacia arriba se interpreta como una base manufacturera interconectada internacionalmente, que absorbe tecnología foránea y reexporta valor agregado. Niveles inferiores a **0,20** son propios de economías más primarizadas.

El capítulo de las flores tiene un GL de **0,06**: comercio interindustrial casi puro. Colombia vende flores y no compra flores. Es exactamente el contraejemplo del café que usa el documento base, y confirma lo que el IBCR del Cuaderno 2 ya insinuaba desde otro ángulo.

El promedio ponderado de la economía colombiana, 0,35, queda por debajo del umbral de 0,45. Colombia comercia menos como eslabón de cadenas globales y más como proveedor de productos terminados o primarios.

## 4.8. Impacto en el negocio y la decisión que habilita

Para una empresa que evalúa dónde instalar una planta de ensamblaje o de manufactura por etapas, un país o clúster con GL alto en la industria relevante es señal de que **ya existe el ecosistema** logístico, arancelario y de proveedores necesario para operar importando insumos y exportando producto terminado.

Para la empresa del caso, el GL de 0,06 en flores dice algo distinto y menos obvio: su negocio **no tiene un eslabón aguas arriba que pueda internacionalizar**. No hay una cadena de valor floral en la que insertarse comprando insumos afuera; hay un producto que se cultiva aquí y se vende afuera. Eso acota las estrategias posibles: la ventaja hay que defenderla en el cultivo y en la logística, no en la integración a una cadena global.

Para un formulador de política comercial, un GL creciente en manufactura es evidencia de que los acuerdos de libre comercio y las zonas francas están atrayendo eslabones de cadenas globales, más allá del simple conteo de exportaciones brutas.

## 4.9. De dónde se saca exactamente el dato

$X_i$ y $M_i$ del mismo código HS —definiendo "industria" según el nivel de agregación elegido— en Trade Map (ITC, 2025). El índice fue formalizado por Grubel y Lloyd (1975) y sigue siendo el estándar para medir comercio intraindustrial (Soo, 2016), con desarrollos posteriores que corrigen sesgos por tamaño de socio comercial (Thom y McDowell, 1999) y que distinguen comercio intraindustrial horizontal de vertical mediante la razón de valores unitarios (Rai et al., 2026).

**Una advertencia sobre el nivel de agregación.** El GL es muy sensible a cuántos dígitos HS se usen. Calculado a 2 dígitos, un país que exporta café e importa trigo aparece como intraindustrial en el capítulo 09 y el 10 por separado, pero si agregáramos ambos en "agricultura" el índice subiría artificialmente. La regla práctica: **cuanto más agregado el nivel, más alto y menos significativo el GL.** Nuestro cálculo a 2 dígitos es, por tanto, un techo: a 6 dígitos los valores serían menores.

---
# 5. El hallazgo que reescribe el caso

Hasta aquí las tres métricas del documento base. Pero al cruzar las dos fuentes apareció algo que ninguna de ellas buscaba, y que cambia la pregunta del caso.

En el Cuaderno 2, Corea del Sur era un **mercado candidato**: un país al que la empresa querría entrar. Los datos aduaneros dicen otra cosa.

In [ ]:
corea_legis = legis[legis["pais_destino"].str.contains("COREA", case=False, na=False)].copy()
total_por_anio = legis.groupby("anio")["valor_fob_usd"].sum()

corea_legis["pct_exportaciones"] = corea_legis.apply(
    lambda f: f["valor_fob_usd"] / total_por_anio[f["anio"]] * 100, axis=1
)

print("Lo que Colombia le exporta a Corea del Sur en claveles frescos (dato DIAN)\n")
print(corea_legis[["anio", "valor_fob_usd", "pct_exportaciones"]].to_string(index=False))

primero = corea_legis[corea_legis["anio"] == 2020]["valor_fob_usd"].iloc[0]
ultimo  = corea_legis[corea_legis["anio"] == 2025]["valor_fob_usd"].iloc[0]
print(f"\n  Crecimiento 2020-2025: {(ultimo / primero - 1) * 100:+.0f} %")

# Y su posicion en el ranking de destinos
ranking_2025 = legis[legis["anio"] == 2025].sort_values("valor_fob_usd", ascending=False).reset_index(drop=True)
puesto = ranking_2025[ranking_2025["pais_destino"].str.contains("COREA", case=False)].index[0] + 1
print(f"  Puesto de Corea del Sur entre los destinos de Colombia en 2025: {puesto} de {len(ranking_2025)}")

Corea del Sur no es un mercado al que entrar. Es el **séptimo destino** de Colombia, con 10,4 millones de dólares y un crecimiento del 210 % en seis años.

Y hay más. Cuando cruzamos lo que Colombia declara haber exportado contra lo que Corea declara haber importado, aparece una diferencia que hay que explicar antes de usar cualquiera de las dos cifras.

In [ ]:
proveedores_ind = leer_socios(ARCHIVOS["kor_prov_ind"])
fila_colombia_kor = proveedores_ind[proveedores_ind["pais"] == "Colombia"].iloc[0]

corea_declara   = float(fila_colombia_kor["Value imported (k$)"]) * 1_000
colombia_declara = ultimo

print("El mismo comercio, declarado por las dos aduanas\n")
print(f"  Colombia declara EXPORTAR a Corea (FOB) : {colombia_declara:>14,.0f} USD")
print(f"  Corea declara IMPORTAR de Colombia (CIF): {corea_declara:>14,.0f} USD")
print(f"\n  Diferencia                              : {corea_declara - colombia_declara:>14,.0f} USD")
print(f"  Sobrecosto implicito                    : {(corea_declara / colombia_declara - 1) * 100:>13.1f} %")

# Y la cuota resultante segun cual de las dos cifras se use
importaciones_corea = mundo_kor * 1_000
print(f"\n  Corea importa en total                  : {importaciones_corea:>14,.0f} USD")
print(f"  Cuota de Colombia segun el dato coreano : {corea_declara / importaciones_corea * 100:>13.2f} %")
print(f"  Cuota si usaramos el FOB colombiano     : {colombia_declara / importaciones_corea * 100:>13.2f} %")

Una diferencia del 60,7 % entre las dos declaraciones. ¿Cuál está mal?

**Ninguna.** Colombia declara el valor **FOB**: lo que vale la mercancía puesta en el aeropuerto de salida. Corea declara el valor **CIF**: eso más el flete y el seguro. Entre Bogotá y Seúl, con flores frescas que viajan refrigeradas en avión, el flete es una fracción enorme del valor.

La prueba de que la explicación es esa y no un error de medición está en el resto de los destinos: para todos los demás países, Trade Map y Legiscomex coinciden casi al dólar, como vimos en la sección 2.7. Solo divergen aquí porque este es el único caso en que comparamos **lo que declara Colombia contra lo que declara el socio**.

Y la consecuencia práctica no es menor: **la cuota de Colombia en el mercado coreano depende de qué cifra uses.** Con el dato coreano, 86,65 %. Con el FOB colombiano contra las importaciones CIF coreanas, 53,9 % — un número que mezcla dos bases de valoración distintas y por eso **está mal**. La comparación correcta es CIF contra CIF: 86,65 %.

In [ ]:
serie_corea = proveedores[proveedores["pais"].isin(["World", "Colombia", "China"])]
anios_serie = [c for c in proveedores.columns if c.isdigit()]

fig, ax = plt.subplots(figsize=(9.5, 5))

mundo_serie = serie_corea.loc[serie_corea["pais"] == "World", anios_serie].iloc[0].astype(float)
for pais, color in [("Colombia", NARANJA), ("China", AZUL)]:
    valores = serie_corea.loc[serie_corea["pais"] == pais, anios_serie].iloc[0].astype(float)
    ax.plot([int(a) for a in anios_serie], valores / mundo_serie * 100,
            color=color, linewidth=2.4, marker="o", markersize=5.5, label=pais, zorder=3)

ax.set_ylabel("Cuota del mercado coreano (%)", fontsize=10, color=GRIS_TEXT)
ax.set_title("Colombia no tiene que entrar a Corea del Sur: ya conquistó ese mercado\n"
             "Cuota de los proveedores de HS 060312 a Corea del Sur, 2016-2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_ylim(0, 100)
ax.set_xticks([int(a) for a in anios_serie])
ax.yaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)
ax.legend(frameon=False, fontsize=9.5, loc="center right")

plt.figtext(0.01, -0.05,
            "El mercado coreano paso de 1,5 a 19,3 millones de USD en nueve anios.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025), declarados por Corea del Sur.",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

**Para pensar.** Los Cuadernos 1 y 2 construyeron un análisis impecable sobre una premisa equivocada: que Corea del Sur era un mercado por conquistar. Ninguna de las métricas de esos cuadernos podía detectar el error, porque todas miraban a Corea del Sur como comprador **del mundo**, nunca como comprador **de Colombia**.

El error se cayó en cuanto se añadió una fuente que desagregaba por origen. No hizo falta una técnica más sofisticada: hizo falta un dato distinto.

¿Cuántas veces, en un análisis real, la métrica es correcta y la premisa es la que falla? ¿Y cómo se protege uno de eso?

---
# 6. El tablero de este cuaderno

Las tres métricas juntas, sobre el caso, más lo que aportó el cruce de fuentes.

In [ ]:
TABLERO = pd.DataFrame([
    ("HHI por destino (clavel, Colombia)", f"{hhi_legiscomex:.4f}",
     f"{numeros_equivalentes(hhi_legiscomex):.1f} destinos equivalentes",
     "Moderadamente concentrado y casi inmovil en seis anios"),
    ("HHI de proveedores (mercado coreano)", f"{hhi_proveedores:.4f}",
     f"{numeros_equivalentes(hhi_proveedores):.1f} proveedores equivalentes",
     "Altamente concentrado: y el que concentra es Colombia"),
    ("HHI por producto (canasta colombiana)", f"{hhi_canasta:.4f}",
     f"{numeros_equivalentes(hhi_canasta):.1f} capitulos equivalentes",
     "97 capitulos en el papel, 6,5 en la practica"),
    ("Theil total (canasta colombiana)", f"{resultado['total']:.4f}",
     f"{resultado['pct_entre']:.0f} % entre secciones",
     "La concentracion es estructural, no de variedad interna"),
    ("Grubel-Lloyd (HS06, flores)", f"{gl_tabla.loc[gl_tabla['codigo'] == '06', 'gl'].iloc[0]:.4f}",
     "Interindustrial puro",
     "No hay cadena de valor floral en la que insertarse"),
    ("Grubel-Lloyd (economia colombiana)", f"{gl_ponderado:.4f}",
     "Por debajo del umbral de 0,45",
     "Colombia no comercia como eslabon de cadenas globales"),
], columns=["Metrica", "Valor", "Lectura directa", "Que significa para la decision"])

print(TABLERO.to_string(index=False))

In [ ]:
# Guardamos los resultados para poder citarlos y reutilizarlos
salidas = {
    "hhi_destinos_serie.csv":     serie_hhi,
    "canasta_hs2_2025.csv":       canasta[["codigo", "producto", "2025", "participacion", "seccion"]],
    "grubel_lloyd_hs2_2025.csv":  gl_tabla[["codigo", "producto", "exportado", "importado", "gl", "ibcr", "seccion"]],
    "theil_por_seccion.csv":      por_seccion,
    "tablero_cuaderno3.csv":      TABLERO,
}

for nombre, tabla in salidas.items():
    ruta = os.path.join(CARPETA_SALIDA, nombre)
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"  guardado: {ruta}  ({len(tabla)} filas)")

# Ficha metodologica: las decisiones que tomamos y hay que poder defender
ficha = pd.DataFrame([
    ("Producto del caso", "HS 060312 - claveles frescos"),
    ("Nivel de agregacion de la canasta", "HS 2 digitos (97 capitulos)"),
    ("Anio de referencia", "2025"),
    ("Vista de Trade Map utilizada", "Serie temporal (no la vista de indicadores)"),
    ("Capitulo excluido del Theil", "HS99 - no clasificado, 0,02 % de la canasta"),
    ("Anio excluido de las series", "2026 - solo cubre enero a junio"),
    ("Fuente del comercio mundial", "Trade Map (ITC, 2025)"),
    ("Fuente del detalle aduanero", "DIAN via Legiscomex (2026), 273.033 declaraciones"),
    ("Valoracion", "Colombia declara FOB; los socios declaran CIF"),
], columns=["decision", "detalle"])

ficha.to_csv(os.path.join(CARPETA_SALIDA, "ficha_metodologica_cuaderno3.csv"),
             index=False, encoding="utf-8-sig")
print(f"\n  guardado: {CARPETA_SALIDA}/ficha_metodologica_cuaderno3.csv")

In [ ]:
# Si estas en Colab y quieres bajar los resultados a tu computador,
# quita el simbolo # de las dos lineas siguientes:

# from google.colab import files
# for nombre in salidas: files.download(os.path.join(CARPETA_SALIDA, nombre))

---
# 7. Los límites de estas métricas y el puente al Cuaderno 4

Las tres métricas de este cuaderno resolvieron tres de las seis preguntas que dejó abierto el Cuaderno 2. Pero comparten una limitación de fondo: **describen la estructura del comercio, no la competitividad.**

| Límite | Qué no pudimos responder |
|---|---|
| **El HHI no dice si eres bueno** | Sabemos que Colombia concentra el 86,65 % del mercado coreano. ¿Eso es ventaja competitiva real o una posición heredada que alguien puede disputar? |
| **El Theil no compara con el mundo** | Sabemos que Colombia concentra en minerales y vegetales. ¿Es más especializada en flores que el promedio mundial, o solo grande en volumen? |
| **El GL no mide eficiencia** | Un GL de 0,06 dice que Colombia no importa flores. No dice si las produce mejor o peor que Ecuador o Kenia. |
| **Ninguna corrige por tamaño de país** | Una economía pequeña puede tener índices extremos solo por ser pequeña. Ninguna de las tres lo corrige. |
| **Ninguna sirve para modelar** | El HHI y el Theil no tienen las propiedades estadísticas necesarias para entrar en una regresión sin distorsión. |

**Lo que viene.** El Cuaderno 4 recorre la familia de índices de **Ventaja Comparativa Revelada**, que es la respuesta de la literatura a exactamente estas preguntas. Se presentan en orden cronológico, porque cada índice nuevo nació explícitamente para corregir una limitación matemática del anterior:

- **Balassa (1965)** — el índice fundacional, y por qué su asimetría lo inutiliza para la estadística.
- **Laursen (2015)** — la corrección simétrica, RSCA.
- **Vollrath (1991)** — integrar las importaciones, para distinguir producción eficiente de maquila.
- **Yu, Cai y Leung (2009)** — el NRCA, con su propiedad de conservación y una trampa numérica que hace fallar el cálculo en silencio.
- **Lafay (1992)** — especialización aislada del ciclo macroeconómico.

Y al final, el cruce **HHI × NRCA**, que el documento base señala como la combinación metodológica más usada en la investigación aplicada de 2024-2026 para decisiones de diversificación de mercados.

---
# 8. Glosario mínimo

**Comercio interindustrial.** Intercambio de bienes de industrias distintas: un país exporta café e importa maquinaria. Es lo que predicen las teorías clásicas del comercio.

**Comercio intraindustrial.** Intercambio de bienes de la misma industria: un país exporta e importa autopartes simultáneamente. Es característico de las cadenas globales de valor.

**Dato espejo.** Cifra de comercio reconstruida combinando lo que declara un país con lo que declaran sus socios. Trade Map es una base de datos espejo.

**Entropía.** Medida de dispersión o "desorden" tomada de la teoría de la información. Es la base matemática del índice de Theil.

**FOB / CIF.** Dos bases de valoración. FOB (*free on board*) es el valor de la mercancía puesta en el punto de salida; CIF (*cost, insurance, freight*) añade flete y seguro. Las exportaciones se declaran FOB y las importaciones CIF, así que **el mismo comercio tiene dos valores distintos** según qué aduana lo reporte.

**Números equivalentes.** El inverso del HHI. Traduce el índice a "cuántos destinos o productos del mismo tamaño equivaldrían a este reparto". Es la forma más clara de comunicarlo.

**Sección del Sistema Armonizado.** Agrupación de capítulos HS en 21 macro-sectores económicos. Es parte de la nomenclatura, no de ninguna base de datos.

**Subpartida NANDINA.** Desagregación a 10 dígitos que usa la Comunidad Andina. El HS 060312 se abre en Colombia en `0603121000` y `0603129000`.

---
# 9. Ejercicios propuestos

**Ejercicio 1 — El HHI de otro producto.** Descarga de Trade Map la lista de mercados importadores para las rosas frescas (HS 060311) exportadas por Colombia y calcula su HHI por destino. ¿Está más o menos concentrado que el clavel? ¿Qué recomendarías si la empresa vendiera los dos?

**Ejercicio 2 — Los números equivalentes en la sala de juntas.** Escribe, en tres frases y sin usar la palabra "índice", cómo le explicarías a un comité directivo el resultado de la sección 2.6. La restricción es real: si no se puede explicar, no se puede usar para decidir.

**Ejercicio 3 — Theil a otro nivel.** Vuelve a calcular la descomposición de Theil agrupando los capítulos en solo tres grupos: primario (secciones I a V), manufactura (VI a XX) y otros (XXI). ¿Sube o baja el componente "entre grupos"? Explica por qué el resultado depende de cómo definas los grupos, y qué implica eso para la honestidad de un informe.

**Ejercicio 4 — El umbral del Grubel-Lloyd.** Filtra `gl_tabla` para quedarte solo con los capítulos que superan 0,45 y suma su comercio. ¿Qué porcentaje del comercio colombiano ocurre en sectores intraindustriales? Compara ese número con el 0,35 del promedio ponderado y explica por qué no dicen lo mismo.

**Ejercicio 5 — La trampa de la agregación.** El GL se calculó a 2 dígitos. Descarga el capítulo 06 a 6 dígitos y recalcula el GL de las flores partida por partida. ¿Sube o baja? Relaciona el resultado con la advertencia de la sección 4.9.

**Ejercicio 6 — Auditar la premisa.** La sección 5 muestra que la premisa del caso era falsa. Escoge otro de los cinco mayores importadores del Cuaderno 2 y verifica, con el archivo de destinos de Legiscomex, cuánto le vende Colombia realmente. ¿Hay más premisas que se caen?

**Ejercicio 7 — FOB contra CIF.** Calcula, para los diez principales destinos, la razón entre lo que declara Colombia y lo que declara el socio. ¿El sobrecosto guarda relación con la distancia? Cruza tu resultado con la columna de distancia del archivo `co_060312_destinos_indicadores.csv`.

---
# Referencias

Arbulú Ballesteros, M., et al. (2026). Competitividad y concentración de mercados en las exportaciones mundiales de maíz. *Revista de Economía Agrícola*.

Chang, C., et al. (2024). Export diversification and creative products: A Theil decomposition approach. *Journal of Cultural Economics*.

Corrales Otazú, J., et al. (2025). Ventaja comparativa revelada y concentración de mercados en las exportaciones de banano de la Comunidad Andina. *Revista Iberoamericana de Comercio Exterior*.

Durán Lima, J. E. (s.f.). *Indicadores de comercio exterior y política comercial: Generalidades metodológicas e indicadores básicos*. Comisión Económica para América Latina y el Caribe.

García Juárez, R., et al. (2025). Concentración de mercados y competitividad en las exportaciones mundiales de café. *Revista de Análisis Económico*.

García Juárez, R., et al. (2026). Índice normalizado de ventaja comparativa revelada y concentración de mercados en las exportaciones de salmón. *Aquaculture Economics & Management*.

Grubel, H. G., & Lloyd, P. J. (1975). *Intra-industry trade: The theory and measurement of international trade in differentiated products*. Macmillan.

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

International Trade Centre. (2025). *Trade Map: Trade statistics for international business development*. https://www.trademap.org

Legiscomex. (2026). *Sistema de inteligencia comercial: Estadísticas de comercio exterior de Colombia* [Base de datos]. Datos primarios de la Dirección de Impuestos y Aduanas Nacionales (DIAN).

McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt & J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a

Rai, S., et al. (2026). Horizontal and vertical intra-industry trade in Nepalese pharmaceutical exports. *South Asian Journal of Trade*.

Serie de recursos — Inteligencia en Negocios Globales. (2026). *Métricas de comercio exterior e inteligencia de negocios globales. Documento 2 de 3 — Nivel intermedio: concentración, comercio intraindustrial y ventaja comparativa revelada*. Universidad EAN.

Soo, K. T. (2016). Intra-industry trade: A Krugman–Ricardo model and data. *Economica, 83*(332), 338–355.

Theil, H. (1967). *Economics and information theory*. North-Holland.

Thom, R., & McDowell, M. (1999). Measuring marginal intra-industry trade. *Weltwirtschaftliches Archiv, 135*(1), 48–61.

Trinh, T., & Nguyen, H. (2021). Export diversification and economic growth: A nonlinear relationship. *Journal of Asian Economics*.

Zhang, Y., et al. (2023). Measuring diversification of creative goods exports with the Theil index. *Creative Industries Journal*.

Harris, C. R., et al. (2020). Array programming with NumPy. *Nature, 585*, 357–362. https://doi.org/10.1038/s41586-020-2649-2